In [7]:
"""
================================================================================
PROJECT: Stats NZ Labour Market Data Warehouse
FILE NAME: 03_reconciliation_audit.py
DESCRIPTION: Auto-discovers raw CSV files across directories/subfolders and 
             reconciles Power BI KPI Cards & Visual Aggregations against 
             the Gold SQLite Data Warehouse.
================================================================================
"""

import os
import sqlite3
import pandas as pd
import numpy as np

# Display numbers with standard commas and 2 decimals (no scientific notation)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------------------------
RAW_DIR = r"D:\STATS NZ DATASET"
DB_PATH = r"D:\ProjectData\LabourMarket_Gold_DW.db"

# Map dataset codes to target CSV filenames
TARGET_FILES = {
    'MEI':  'employment-indicators-july-2026-csv-tables.csv',
    'HLFS': 'hlfs-jun26qtr-csv.csv',
    'LCI':  'lci-jun26qtr-csv.csv',
    'LMS':  'lms-jun26qtr-tables.csv',
    'QES':  'qes-jun26qtr-csv.csv'
}

print("=" * 85)
print("🚀 STARTING END-TO-END RECONCILIATION AUDIT (RAW CSV VS. STAR SCHEMA)")
print("=" * 85)

# Dynamic file discovery: scan RAW_DIR for missing paths
discovered_files = {}
for root, dirs, files in os.walk(RAW_DIR):
    for code, target_name in TARGET_FILES.items():
        if target_name.lower() in [f.lower() for f in files]:
            discovered_files[code] = os.path.join(root, target_name)



🚀 STARTING END-TO-END RECONCILIATION AUDIT (RAW CSV VS. STAR SCHEMA)


In [8]:
# ------------------------------------------------------------------------------
# 2. LOAD RAW CSV DATA & CALCULATE RAW BENCHMARKS
# ------------------------------------------------------------------------------
print("\n📂 [STEP 1] Extracting & Aggregating Raw Source Data...")

raw_summary_list = []
for code, target_name in TARGET_FILES.items():
    filepath = discovered_files.get(code)
    
    if filepath and os.path.exists(filepath):
        df_raw = pd.read_csv(filepath, low_memory=False)
        
        # Standardize column headers to uppercase
        df_raw.columns = [c.strip().upper() for c in df_raw.columns]
        
        # Detect numeric column (DATA_VALUE or VALUE)
        val_col = 'DATA_VALUE' if 'DATA_VALUE' in df_raw.columns else 'VALUE' if 'VALUE' in df_raw.columns else None
        
        if val_col:
            df_raw['CleanValue'] = pd.to_numeric(df_raw[val_col], errors='coerce')
            df_clean = df_raw.dropna(subset=['CleanValue'])
            
            raw_summary_list.append({
                'DatasetCode': code,
                'Raw_Total_Rows': len(df_raw),
                'Raw_Clean_Rows': len(df_clean),
                'Raw_Sum_DataValue': df_clean['CleanValue'].sum(),
                'Raw_Avg_DataValue': df_clean['CleanValue'].mean()
            })
            print(f"  ✓ Successfully processed {code} ({os.path.basename(filepath)}): {len(df_clean):,} valid rows")
        else:
            print(f"⚠️ Warning: Value column missing in {filepath}")
    else:
        print(f"⚠️ Warning: Could not locate file '{target_name}' under {RAW_DIR}")

raw_df = pd.DataFrame(raw_summary_list)




📂 [STEP 1] Extracting & Aggregating Raw Source Data...
  ✓ Successfully processed MEI (employment-indicators-july-2026-csv-tables.csv): 34,532 valid rows
  ✓ Successfully processed HLFS (hlfs-jun26qtr-csv.csv): 1,207,728 valid rows
  ✓ Successfully processed LCI (lci-jun26qtr-csv.csv): 32,926 valid rows
  ✓ Successfully processed LMS (lms-jun26qtr-tables.csv): 1,438,668 valid rows
  ✓ Successfully processed QES (qes-jun26qtr-csv.csv): 198,014 valid rows


In [9]:
# ------------------------------------------------------------------------------
# 3. QUERY STAR SCHEMA WAREHOUSE & CALCULATE DW BENCHMARKS
# ------------------------------------------------------------------------------
print("\n🗄️ [STEP 2] Extracting Aggregations from SQLite Gold Data Warehouse...")

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"❌ Database file not found at: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)

dw_query = """
SELECT 
    d.DatasetCode,
    COUNT(f.FactID) AS DW_Total_Rows,
    SUM(f.DataValue) AS DW_Sum_DataValue,
    AVG(f.DataValue) AS DW_Avg_DataValue
FROM FactLabourMarket f
JOIN DimDataset d ON f.DatasetKey = d.DatasetKey
GROUP BY d.DatasetCode
"""
dw_df = pd.read_sql(dw_query, conn)




🗄️ [STEP 2] Extracting Aggregations from SQLite Gold Data Warehouse...


In [10]:
# ------------------------------------------------------------------------------
# 4. KPI CARD RECONCILIATION AUDIT
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("📊 [AUDIT 1] POWER BI OVERALL KPI CARD RECONCILIATION")
print("=" * 85)

audit_kpi = pd.merge(raw_df, dw_df, on='DatasetCode')
audit_kpi['Row_Diff'] = audit_kpi['DW_Total_Rows'] - audit_kpi['Raw_Clean_Rows']
audit_kpi['Sum_Diff'] = audit_kpi['DW_Sum_DataValue'] - audit_kpi['Raw_Sum_DataValue']
audit_kpi['Status'] = np.where((audit_kpi['Row_Diff'] == 0) & (abs(audit_kpi['Sum_Diff']) < 0.01), '✅ MATCH', '❌ MISMATCH')

print(audit_kpi[['DatasetCode', 'Raw_Clean_Rows', 'DW_Total_Rows', 'Raw_Sum_DataValue', 'DW_Sum_DataValue', 'Status']].to_string(index=False))




📊 [AUDIT 1] POWER BI OVERALL KPI CARD RECONCILIATION
DatasetCode  Raw_Clean_Rows  DW_Total_Rows  Raw_Sum_DataValue  DW_Sum_DataValue  Status
        MEI           34532          34532      5208798090.98     5208798090.98 ✅ MATCH
       HLFS         1207728        1207728       206960545.80      206960545.80 ✅ MATCH
        LCI           32926          32926        28491939.40       28491939.40 ✅ MATCH
        LMS         1438668        1438668    877848757685.52   877848757685.52 ✅ MATCH
        QES          198014         198014    877613305200.32   877613305200.32 ✅ MATCH


In [11]:
# ------------------------------------------------------------------------------
# 5. POWER BI VISUAL RECONCILIATION: YEARLY TREND LINE CHART
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("📈 [AUDIT 2] POWER BI VISUAL COMPARISON: YEARLY TRENDS (RECENT 5 YEARS)")
print("=" * 85)

dw_yearly_query = """
SELECT 
    dt.Year,
    d.DatasetCode,
    SUM(f.DataValue) AS DW_Yearly_Sum,
    AVG(f.DataValue) AS DW_Yearly_Avg
FROM FactLabourMarket f
JOIN DimDate dt ON f.DateKey = dt.DateKey
JOIN DimDataset d ON f.DatasetKey = d.DatasetKey
WHERE dt.Year >= 2022
GROUP BY dt.Year, d.DatasetCode
ORDER BY dt.Year DESC, d.DatasetCode;
"""
dw_yearly = pd.read_sql(dw_yearly_query, conn)
print(dw_yearly.to_string(index=False))




📈 [AUDIT 2] POWER BI VISUAL COMPARISON: YEARLY TRENDS (RECENT 5 YEARS)
Year DatasetCode  DW_Yearly_Sum  DW_Yearly_Avg
2026        HLFS     4579589.20         175.94
2026         LCI      948729.59        1210.11
2026         LMS 27229373053.13      921779.72
2026         MEI   214768062.89       98247.06
2026         QES 27223844734.34     9986736.88
2025        HLFS     8963654.20         173.56
2025         LCI     1869392.82        1192.21
2025         LMS 53091118864.73      904987.96
2025         MEI   367770243.84       98019.79
2025         QES 53080285817.71     9735929.17
2024        HLFS     8962001.70         173.50
2024         LCI     1823805.61        1163.14
2024         LMS 51560900516.90      878754.16
2024         MEI   372233860.81       99209.45
2024         QES 51550114709.59     9455266.82
2023        HLFS     8893308.30         172.81
2023         LCI     1753021.05        1118.00
2023         LMS 49185120740.55      841001.31
2023         MEI   372426781.10    

In [12]:
# ------------------------------------------------------------------------------
# 6. POWER BI DAX MEASURE COMPARISON SIMULATION (HLFS DATASET BENCHMARK)
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("🧮 [AUDIT 3] POWER BI DAX MEASURE PRE-CALCULATIONS (HLFS DATASET BENCHMARK)")
print("=" * 85)

hlfs_dax_query = """
SELECT 
    dt.Year,
    SUM(f.DataValue) AS Total_DataValue,
    AVG(f.DataValue) AS Average_DataValue
FROM FactLabourMarket f
JOIN DimDate dt ON f.DateKey = dt.DateKey
JOIN DimDataset d ON f.DatasetKey = d.DatasetKey
WHERE d.DatasetCode = 'HLFS' AND dt.Year >= 2020
GROUP BY dt.Year
ORDER BY dt.Year ASC;
"""
hlfs_trend = pd.read_sql(hlfs_dax_query, conn)

hlfs_trend['Prior_Year_Value'] = hlfs_trend['Total_DataValue'].shift(1)
hlfs_trend['YoY_Growth_Amount'] = hlfs_trend['Total_DataValue'] - hlfs_trend['Prior_Year_Value']
hlfs_trend['YoY_Growth_Pct'] = (hlfs_trend['YoY_Growth_Amount'] / hlfs_trend['Prior_Year_Value']) * 100

print(hlfs_trend[['Year', 'Total_DataValue', 'Prior_Year_Value', 'YoY_Growth_Amount', 'YoY_Growth_Pct']].to_string(index=False))

conn.close()

print("\n" + "=" * 85)
print("✅ RECONCILIATION COMPLETE! SAVE THESE BENCHMARKS TO VERIFY POWER BI VISUALS.")
print("=" * 85)


🧮 [AUDIT 3] POWER BI DAX MEASURE PRE-CALCULATIONS (HLFS DATASET BENCHMARK)
Year  Total_DataValue  Prior_Year_Value  YoY_Growth_Amount  YoY_Growth_Pct
2020       8464634.00               NaN                NaN             NaN
2021       8567811.00        8464634.00          103177.00            1.22
2022       8669161.00        8567811.00          101350.00            1.18
2023       8893308.30        8669161.00          224147.30            2.59
2024       8962001.70        8893308.30           68693.40            0.77
2025       8963654.20        8962001.70            1652.50            0.02
2026       4579589.20        8963654.20        -4384065.00          -48.91

✅ RECONCILIATION COMPLETE! SAVE THESE BENCHMARKS TO VERIFY POWER BI VISUALS.
